In [ ]:
from model import MobilnetSegRot
import torch
import torch.nn as nn
from utils import DEFAULT_TRANSFORM
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import numpy as np
from glob import glob

# Cargar modelo
model = MobilnetSegRot(num_classes=72)
model.load_state_dict(torch.load('Mobilnet_mask_rot.pth', map_location='cpu'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
images = glob('../Data_img/Test/*.jpg')

with torch.no_grad():
    for img_path in images[:10]:
        img = Image.open(img_path).convert('RGB')
        img_original = img.copy()
        img_trans = DEFAULT_TRANSFORM(img).unsqueeze(0).to(device)

        # Inferencia
        mask, angle = model(img_trans)
        predicted_mask = (mask > 0.5).float()
        predicted_mask = nn.functional.interpolate(
            predicted_mask, size=img.size[::-1], mode='bilinear', align_corners=False
        )

        angle = angle.argmax(dim=1).item() * 5
        print("Ángulo predicho (grados):", angle)

        img = img.rotate(-angle)
        predicted_mask = Image.fromarray((predicted_mask.squeeze().cpu().numpy() * 255).astype(np.uint8))
        predicted_mask = predicted_mask.rotate(-angle)
        predicted_mask = np.array(predicted_mask)
        img_np = np.array(img)

        contours, _ = cv2.findContours(predicted_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        img_with_rect = img_np.copy()
        warped = None

        if contours:
            cnt = max(contours, key=cv2.contourArea)
            rect = cv2.minAreaRect(cnt)
            box = cv2.boxPoints(rect)
            box = np.intp(box)
            cv2.drawContours(img_with_rect, [box], 0, (255, 0, 0), 3)
        
            # Puntos fuente
            src_pts = box.astype("float32")
        
            # Ordenar consistentemente los puntos: [top-left, top-right, bottom-right, bottom-left]
            rect_pts = np.zeros((4, 2), dtype="float32")
            s = src_pts.sum(axis=1)
            rect_pts[0] = src_pts[np.argmin(s)]  # top-left
            rect_pts[2] = src_pts[np.argmax(s)]  # bottom-right
        
            diff = np.diff(src_pts, axis=1)
            rect_pts[1] = src_pts[np.argmin(diff)]  # top-right
            rect_pts[3] = src_pts[np.argmax(diff)]  # bottom-left
        
            # Calcular dimensiones en base a distancias
            W = int(np.linalg.norm(rect_pts[0] - rect_pts[1]))
            H = int(np.linalg.norm(rect_pts[1] - rect_pts[2]))
        
            if W > 0 and H > 0:
                dst_pts = np.array([
                    [0, 0],
                    [W-1, 0],
                    [W-1, H-1],
                    [0, H-1]
                ], dtype="float32")
        
                # Transformación de perspectiva
                M = cv2.getPerspectiveTransform(rect_pts, dst_pts)
                warped = cv2.warpPerspective(img_np, M, (W, H))

        # Visualización en matriz 2x2
        plt.figure(figsize=(15, 15))
        
        # Imagen original (posición 1)
        plt.subplot(2, 2, 1)
        plt.title("Imagen original", fontsize=14, fontweight='bold')
        plt.imshow(img_original)
        plt.axis('off')

        # Máscara predicha (posición 2)
        plt.subplot(2, 2, 2)
        plt.title("Máscara predicha", fontsize=14, fontweight='bold')
        plt.imshow(predicted_mask, cmap='gray')
        plt.axis('off')

        # Rectángulo detectado (posición 3)
        plt.subplot(2, 2, 3)
        plt.title("Rectángulo detectado", fontsize=14, fontweight='bold')
        plt.imshow(img_with_rect)
        plt.axis('off')

        # Recorte transformado (posición 4)
        plt.subplot(2, 2, 4)
        if warped is not None:
            plt.title("Recorte transformado", fontsize=14, fontweight='bold')
            plt.imshow(warped)
            print("Dimensiones del recorte:", warped.shape)
        else:
            plt.title("Sin recorte disponible", fontsize=14, fontweight='bold')
            plt.text(0.5, 0.5, 'No se pudo\ngenerar recorte', 
                    ha='center', va='center', transform=plt.gca().transAxes,
                    fontsize=12, bbox=dict(boxstyle="round", facecolor='lightgray'))
        plt.axis('off')

        plt.tight_layout()
        plt.show()